# 04 · Artefactos y lo que falta para hablar de oportunidad

**Objetivo:** Medir cuánto de una diferencia de precio entre municipios se mantiene cuando se comparan productos, presentaciones y periodos equivalentes, y distinguir lo que los datos públicos permiten afirmar de lo que haría falta para evaluar una oportunidad de negocio.

**Pregunta:** Para las que sobreviven, ¿qué información haría falta para saber si son una oportunidad, y cuál no está en los datos públicos? ¿Qué debe revisar una empresa antes de tomar una decisión comercial basada en una diferencia de precio?

Por qué importa: los ejemplos concretos muestran cómo nace una diferencia falsa; la tabla final separa lo
que dicen los precios públicos de lo que haría falta para decidir.

**Decisión:** los casos de artefacto se eligen con una regla y no a mano: en cada paso, el nombre de producto cuya
diferencia más baja con esa validación. Alternativa descartada: escoger ejemplos «bonitos», que exagerarían el efecto.

In [1]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "1")
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import entorno
entorno.setup()
import profeco, tablas, estadistica as E, charts as C
import embudo as M
PROC, META, CHARTS, SOCIAL = entorno.PROC, entorno.META, entorno.CHARTS, entorno.SOCIAL
CFG = entorno.config()
def reg(*a, **k):
    entorno.registrar(*a, **k)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)
raw = entorno.load_raw()
d_all = M.prep(raw)
d = d_all[~d_all.catalogo.isin(M.FUERA_PRINCIPAL)].copy()      # universo principal
params = json.loads((META / "parametros.json").read_text(encoding="utf-8"))
REPS = params.get("bootstrap_replicas", 500)
C.register_cities(d_all.municipio_id.unique())
res, prod, un, pm = M.embudo(d)
U = set(prod.index)
F = "04_artefactos_y_lo_que_falta"
pasos = list(prod.columns)
caida = pd.DataFrame({p: prod[pasos[i - 1]] - prod[p] for i, p in enumerate(pasos) if i > 0})
display(caida.describe().round(1))

def cerca(g, q):
    return g.iloc[(g.precio - g.precio.quantile(q)).abs().argmin()]

def mezcla(producto, col, mun):
    x = d[(d.producto == producto) & (d.municipio_id == mun)]
    s = x.groupby(col).precio_num.agg(["size", "mean"]).sort_values("size", ascending=False)
    top = s.index[0]
    return top, s.loc[top, "size"] / s["size"].sum() * 100, s.loc[top, "mean"], x.precio_num.mean()

def texto_val(v, col):
    if col == "pres_key":
        return M.presentacion_corta(v)
    if col == "marca":
        return M.marca_corta(v) or "sin marca"
    return M.giro_corto(v)

,presentacion,marca,giro,tiendas,periodo,extremos
count,279.0,279.0,279.0,279.0,279.0,279.0
mean,19.0,1.6,-2.9,11.1,0.0,0.1
std,64.1,7.8,15.7,20.1,0.1,0.5
min,-36.3,-27.6,-90.5,-23.1,-0.4,-0.0
25%,0.0,0.0,-3.0,1.4,-0.1,0.0
50%,0.0,0.0,0.0,4.5,-0.0,0.0
75%,12.1,1.8,2.0,14.9,0.1,0.0
max,518.9,47.4,30.8,161.2,0.5,6.7


## 1. Cinco casos donde la diferencia ingenua era un artefacto

In [2]:
casos = []
# Presentación, marca y giro: el municipio caro y el barato típicos registran mezclas distintas del mismo nombre.
for paso, col, tipo in [("presentacion", "pres_key", "Presentación distinta"), ("marca", "marca", "Otra marca"),
                        ("giro", "giro", "Otro giro")]:
    # El nombre que más baja con esta validación y cuyo municipio caro y barato típicos registran mezclas distintas.
    for p in caida[paso].sort_values(ascending=False).index[:25]:
        g = pm["ingenua"][pm["ingenua"].producto == p]
        lo, hi = cerca(g, .1), cerca(g, .9)
        tl, sl, ml, al = mezcla(p, col, lo.municipio_id)
        th, sh, mh, ah = mezcla(p, col, hi.municipio_id)
        if tl != th:
            break
    expl = (f"En {C.city(hi.municipio_id)} el promedio es {C.money(ah)} y {sh:.0f}% de los registros son "
            f"{texto_val(th, col)}; en {C.city(lo.municipio_id)} es {C.money(al)} y {sl:.0f}% son {texto_val(tl, col)}.")
    casos.append({"tipo": tipo, "producto": p, "ingenua": prod.loc[p, "ingenua"], "antes": prod.loc[p, pasos[pasos.index(paso) - 1]],
                  "despues": prod.loc[p, paso], "validada": prod.loc[p, "extremos"], "explicacion": expl})
# Una sola tienda: el artículo del paso «giro» cuya brecha más baja al exigir 3 tiendas.
ug, ut = un["giro"], un["tiendas"]
m = ug.merge(ut[M.UNIDAD + ["brecha_pct"]], on=M.UNIDAD, how="left", suffixes=("", "_t"))
m = m[m.producto.isin(U)]
m["caida"] = m.brecha_pct - m.brecha_pct_t.fillna(np.nan)
r1 = m.sort_values("caida", ascending=False).iloc[0]
key = (pm["giro"][M.UNIDAD] == r1[M.UNIDAD].values).all(axis=1)
g = pm["giro"][key]
g1 = g[g.tiendas == 1]
hi = g1.iloc[np.abs(np.log(g1.precio / g.precio.median())).argmax()]
ob = d[(d.municipio_id == hi.municipio_id) & (d[M.UNIDAD] == r1[M.UNIDAD].values).all(axis=1)]
cad = C.chain(ob.cadena_comercial.mode()[0])
nom = f"{r1.producto} {M.marca_corta(r1.marca)}".strip()
expl = (f"{nom}: en {C.city(hi.municipio_id)} el «precio del municipio» era el de una sola tienda ({cad}, "
        f"{C.money(hi.precio)}, contra {C.money(g.precio.median())} en el municipio típico); con 3 tiendas o más, la "
        f"diferencia del artículo baja de {r1.brecha_pct:.0f}% a {r1.brecha_pct_t:.0f}%.")
casos.append({"tipo": "Una sola tienda", "producto": r1.producto, "ingenua": prod.loc[r1.producto, "ingenua"],
              "antes": r1.brecha_pct, "despues": r1.brecha_pct_t, "validada": prod.loc[r1.producto, "extremos"], "explicacion": expl})
# Captura errónea: el posible error con mayor razón frente a su cadena entre los artículos del universo.
spe = M.marcar_errores(M._tiendas(d))
e = spe[spe.posible_error & spe.producto.isin(U) & (spe.tiendas_mun >= 3)].copy()
e["dist"] = np.abs(np.log(e.razon_cadena))
e1 = e.sort_values("dist", ascending=False).iloc[0]
med = e1.precio / e1.razon_cadena
veces = f"{1 / e1.razon_cadena:.0f} veces menos que" if e1.razon_cadena < 1 else f"{e1.razon_cadena:.0f} veces"
expl = (f"{M.etiqueta(e1.producto, e1.pres_key, e1.marca, e1.presentacion)}: una tienda {C.chain(e1.cadena_comercial)} en "
        f"{C.city(e1.municipio_id)} registró {C.money(e1.precio)}, {veces} la mediana de su cadena ({C.money(med)}). "
        "Se marca y se excluye; no se borra del archivo.")
casos.append({"tipo": "Captura errónea", "producto": e1.producto, "ingenua": prod.loc[e1.producto, "ingenua"],
              "antes": prod.loc[e1.producto, "periodo"], "despues": prod.loc[e1.producto, "extremos"],
              "validada": prod.loc[e1.producto, "extremos"], "explicacion": expl})
casos = pd.DataFrame(casos)
display(casos.round(1))
casos.to_csv(PROC / "casos_artefacto.csv", index=False, encoding="utf-8-sig")
for i, c in casos.iterrows():
    reg(f"art{i + 1}_ingenua", c.ingenua, fuente=F, nota=f"{c.tipo}: {c.producto}")
    reg(f"art{i + 1}_validada", c.validada, fuente=F, nota=f"{c.tipo}: {c.producto}")
    reg(f"art{i + 1}_antes", c.antes, fuente=F)
    reg(f"art{i + 1}_despues", c.despues, fuente=F)
    reg(f"art{i + 1}_producto", c.producto, fuente=F)
    reg(f"art{i + 1}_explicacion", c.explicacion, fuente=F)

,tipo,producto,ingenua,antes,despues,validada,explicacion
0,Presentación distinta,Almendras,537.9,537.9,19.0,10.6,En Monterrey el promedio es $232 y 67% de los registros son 1 kg; ...
1,Otra marca,Ajonjolí,205.7,40.7,5.1,5.3,En Juárez el promedio es $64.79 y 57% de los registros son sin mar...
2,Otro giro,Cinta Adhesiva,40.7,55.6,29.3,4.7,En Naucalpan de Juárez el promedio es $37.93 y 57% de los registro...
3,Una sola tienda,Basa,42.8,71.3,3.2,31.8,Basa: en Puebla el «precio del municipio» era el de una sola tiend...
4,Captura errónea,Carne Pollo,49.7,7.2,7.2,7.2,"Carne Pollo (1 kg, granel, muslo (con piel)): una tienda Walmart e..."


## 2. Qué haría falta para hablar de oportunidad

Una diferencia de precio al consumidor que sobrevive a la validación es una **pregunta**, no una oportunidad. Para
saber si conviene actuar hacen falta datos que el archivo de PROFECO no tiene. **No se calcula arbitraje ni
rentabilidad**: sin costos ni márgenes, cualquier cifra sería un supuesto. Las fuentes de INEGI se mencionan solo como
siguiente paso; no se usaron.

In [3]:
falta = pd.DataFrame([
    ["Costos de adquisición y logística", "No", "Nada: PROFECO registra solo el precio de anaquel",
     "Cotizaciones de proveedores, tarifas de flete y distancias a centros de distribución (datos propios)"],
    ["Márgenes", "No", "Nada: sin costo no hay margen",
     "Estados financieros por plaza o categoría (datos propios)"],
    ["Demanda y volumen", "No", "Nada: no hay ventas ni piezas vendidas",
     "Ventas propias por plaza; como siguiente paso, gasto de los hogares por región (INEGI, ENIGH)"],
    ["Competencia y características del mercado", "Parcial", "Cadenas y tiendas monitoreadas en cada municipio",
     "Número de establecimientos por municipio y giro (INEGI, DENUE y Censos Económicos)"],
    ["Tamaño de la población", "No", "Nada en el archivo de precios",
     "Población por municipio (INEGI, Censo de Población y Vivienda)"],
    ["Precio en tiendas no monitoreadas", "No", "Solo tiendas que visita PROFECO",
     "Levantamiento propio de precios en la plaza"],
    ["Estabilidad en el tiempo", "Parcial", "Una quincena; el precio casi no cambia entre semanas",
     "Varias quincenas de PROFECO para ver si la diferencia persiste"],
], columns=["Qué se necesitaría", "¿Está en los datos usados?", "Qué sí hay", "Dónde podría buscarse (siguiente paso)"])
display(falta)
falta.to_csv(PROC / "que_falta.csv", index=False, encoding="utf-8-sig")
reg("falta_no", (falta["¿Está en los datos usados?"] == "No").sum(), n=len(falta), fuente=F)

,Qué se necesitaría,¿Está en los datos usados?,Qué sí hay,Dónde podría buscarse (siguiente paso)
0,Costos de adquisición y logística,No,Nada: PROFECO registra solo el precio de anaquel,"Cotizaciones de proveedores, tarifas de flete y distancias a centr..."
1,Márgenes,No,Nada: sin costo no hay margen,Estados financieros por plaza o categoría (datos propios)
2,Demanda y volumen,No,Nada: no hay ventas ni piezas vendidas,"Ventas propias por plaza; como siguiente paso, gasto de los hogare..."
3,Competencia y características del mercado,Parcial,Cadenas y tiendas monitoreadas en cada municipio,"Número de establecimientos por municipio y giro (INEGI, DENUE y Ce..."
4,Tamaño de la población,No,Nada en el archivo de precios,"Población por municipio (INEGI, Censo de Población y Vivienda)"
5,Precio en tiendas no monitoreadas,No,Solo tiendas que visita PROFECO,Levantamiento propio de precios en la plaza
6,Estabilidad en el tiempo,Parcial,Una quincena; el precio casi no cambia entre semanas,Varias quincenas de PROFECO para ver si la diferencia persiste


## 3. Qué revisar antes de decidir con una diferencia de precio

1. **¿Es el mismo artículo?** Misma presentación, marca y canal. La presentación y la marca explican buena parte de la
   diferencia ingenua.
2. **¿Cuántas tiendas hay detrás?** Con una o dos tiendas, el «precio del municipio» es el de esas tiendas.
3. **¿Supera al azar?** Si repartir las tiendas al azar entre municipios da una diferencia parecida, no hay diferencia
   de lugar.
4. **¿La misma cadena cobra distinto?** Si sí, es una decisión de precio por zona; si no, es mezcla de cadenas.
5. **¿Persiste?** Repetir con otra quincena antes de actuar.
6. **¿Qué cuesta llegar ahí y cuánto se vende?** Sin costos, márgenes y volumen, la diferencia no dice si conviene
   actuar.

In [4]:
v = entorno.cifras()
display(Markdown(f'''**Conclusión.** Los artefactos más grandes vienen de mezclar presentaciones y marcas bajo un mismo
nombre: en «{casos.producto[0]}», la diferencia ingenua de {casos.ingenua[0]:.0f}% baja a {casos.validada[0]:.0f}% al
comparar lo mismo. De las {len(falta)} piezas de información que harían falta para hablar de oportunidad,
{v["falta_no"]:.0f} no están en los datos públicos usados y {len(falta) - v["falta_no"]:.0f} solo en parte.'''))

**Conclusión.** Los artefactos más grandes vienen de mezclar presentaciones y marcas bajo un mismo
nombre: en «Almendras», la diferencia ingenua de 538% baja a 11% al
comparar lo mismo. De las 7 piezas de información que harían falta para hablar de oportunidad,
5 no están en los datos públicos usados y 2 solo en parte.